In [9]:
# =========================
# Titanic Logistic Regression
# =========================

import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

# -----------------------------
# 1. 데이터 로드
# -----------------------------
titanic = sns.load_dataset("titanic")

# -----------------------------
# 2. 필요한 컬럼 선택
# -----------------------------
df = titanic[["survived", "age", "fare", "sex", "pclass"]].copy()

# -----------------------------
# 3. 결측치 처리
# -----------------------------
df["age"] = df["age"].fillna(df["age"].median())

# -----------------------------
# 4. One-Hot Encoding
# -----------------------------
df = pd.get_dummies(df, columns=["sex"], drop_first=True)

df["sex_male"] = df["sex_male"].astype(int)
print(df)

# -----------------------------
# 5. X, y 분리 (핵심 수정)
# -----------------------------
X = df[["age", "fare", "pclass", "sex_male"]]
y = df["survived"]

# -----------------------------
# 6. train / test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=100
)

# -----------------------------
# 7. 모델 생성
# -----------------------------
model = LogisticRegression(max_iter=1000)

# -----------------------------
# 8. 학습
# -----------------------------
model.fit(X_train, y_train)

# -----------------------------
# 9. 예측
# -----------------------------
y_pred = model.predict(X_test)

# -----------------------------
# 10. 평가
# -----------------------------
print("정확도")
print(accuracy_score(y_test, y_pred))

print("\n혼동행렬")
print(confusion_matrix(y_test, y_pred))

print("\n분류 리포트")
print(classification_report(y_test, y_pred))

     survived   age     fare  pclass  sex_male
0           0  22.0   7.2500       3         1
1           1  38.0  71.2833       1         0
2           1  26.0   7.9250       3         0
3           1  35.0  53.1000       1         0
4           0  35.0   8.0500       3         1
..        ...   ...      ...     ...       ...
886         0  27.0  13.0000       2         1
887         1  19.0  30.0000       1         0
888         0  28.0  23.4500       3         0
889         1  26.0  30.0000       1         1
890         0  32.0   7.7500       3         1

[891 rows x 5 columns]
정확도
0.7932960893854749

혼동행렬
[[90 14]
 [23 52]]

분류 리포트
              precision    recall  f1-score   support

           0       0.80      0.87      0.83       104
           1       0.79      0.69      0.74        75

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



In [10]:
import statsmodels.api as sm

# -------------------------
# 1. X, y 준비
# -------------------------
X = df[["age", "fare", "pclass", "sex_male"]]
y = df["survived"]

# -------------------------
# 2. 상수항 추가 (필수)
# -------------------------
X_sm = sm.add_constant(X)

# -------------------------
# 3. Logit 모델
# -------------------------
model = sm.Logit(y, X_sm)
result = model.fit()

# -------------------------
# 4. 결과 출력
# -------------------------
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.452022
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  891
Model:                          Logit   Df Residuals:                      886
Method:                           MLE   Df Model:                            4
Date:                Tue, 16 Jun 2026   Pseudo R-squ.:                  0.3212
Time:                        15:41:06   Log-Likelihood:                -402.75
converged:                       True   LL-Null:                       -593.33
Covariance Type:            nonrobust   LLR p-value:                 3.282e-81
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.6553      0.509      9.153      0.000       3.659       5.652
age           -0.0331      0.

# 🚢 Titanic Logistic Regression 결과 해석

## 📊 1. 전체 모델 상태

### 🔥 모델 적합도
- Pseudo R² = **0.3212**
- Log-Likelihood = **-402.75**
- LL-Null = **-593.33**

👉 해석:
- 아무 변수도 없는 모델보다 훨씬 좋아짐
- 약 **32% 설명력**
- Titanic 데이터 기준으로 **꽤 좋은 모델**

---

### 🔥 LLR p-value
- p-value = **3.282e-81**

👉 해석:
- 모델 전체가 **통계적으로 매우 유의미함**
- “이 모델은 의미 있다”라고 강하게 말할 수 있음

---

# 📊 2. 변수별 해석

## 🔵 1) sex_male (가장 중요한 변수)

- coef = **-2.6073**
- p-value < 0.001

### 👉 해석
- 남성일수록 생존 확률 크게 감소
- 매우 강력한 영향 변수

### 🔥 odds ratio

exp(-2.6073) ≈ 0.07


👉 의미:
- 남성은 여성 대비 생존 확률이 약 **93% 감소**

---

## 🔵 2) pclass (객실 등급)

- coef = **-1.1529**
- p-value < 0.001

### 👉 해석
- 등급 숫자가 커질수록 (3등급) 생존 확률 감소
- 1등급 > 2등급 > 3등급 구조

---

## 🔵 3) age (나이)

- coef = **-0.0331**
- p-value < 0.001

### 👉 해석
- 나이가 많을수록 생존 확률 감소
- 영향은 비교적 완만함

---

## 🔵 4) fare (운임)

- coef = **0.0006**
- p-value = **0.771**

### 👉 해석
- 생존에 거의 영향 없음
- 통계적으로 유의하지 않음 (중요하지 않은 변수)

---

## 🔵 5) const (절편)

- coef = **4.6553**

👉 의미:
- 기본 생존 확률의 baseline 역할

---

# 📊 3. 변수 영향력 순위


1️⃣ sex_male (가장 중요)
2️⃣ pclass
3️⃣ age
4️⃣ fare (거의 영향 없음)


---

# 🧠 4. 전체 직관

👉 Titanic 생존 패턴:

- 남성 → 생존 확률 낮음
- 높은 객실 등급 → 생존 확률 높음
- 나이 많을수록 → 생존 확률 감소
- 요금(fare)은 영향 거의 없음

---

# 📈 5. odds ratio 해석

| 변수 | odds ratio | 의미 |
|------|------------|------|
| sex_male | 0.07 | 남성 생존 확률 약 93% 감소 |
| pclass | 0.31 | 등급 낮을수록 생존 감소 |
| age | 0.97 | 나이 증가 시 생존 감소 |

---

# ⚡ 6. 핵심 요약


Titanic 생존은 성별이 가장 강력한 영향 변수이며,
그 다음이 객실 등급이다.


---

# 🚀 7. 결론

- 가장 중요한 변수: **sex_male**
- 모델 설명력: **약 32%**
- 전체 모델: **통계적으로 매우 유의미**

# 📊 오즈(Odds) 개념 정리

오즈(Odds)는 어떤 사건이 발생할 확률과 발생하지 않을 확률의 비율을 의미하는 통계적 개념임.  
일상적인 확률(Probability)과 혼동되기 쉬우나, 데이터 분석과 머신러닝에서 핵심적으로 사용되는 지표임.

---

# 1. 확률과 오즈의 차이

## 📌 확률 (Probability)
전체 경우의 수 중 특정 사건이 발생할 비율임.

**공식**

P = (특정 사건 발생 경우) / (전체 경우)

---

## 📌 오즈 (Odds)
특정 사건이 발생할 확률을 발생하지 않을 확률로 나눈 값임.

**공식**

Odds = P / (1 - P)

---

## 📌 예시 (주사위에서 6이 나오는 경우)

- 확률:

1 / 6

- 오즈:

(1 / 6) / (5 / 6) = 1 / 5

👉 의미:  
“6이 발생할 확률 대비 발생하지 않을 확률의 비율이 1:5임”

---

# 2. 확률을 오즈로 변환하는 이유

오즈는 성공 대비 실패의 비율을 나타내는 지표로서, 사건의 상대적 가능성 판단에 유리한 형태임.

---

## 📊 확률과 오즈 비교

| 확률 (P) | 오즈 (Odds) | 의미 |
|----------|-------------|------|
| 0.5 | 1.0 | 성공과 실패가 동일한 수준임 |
| 0.1 | 0.11 | 실패 가능성이 매우 높은 상태임 |
| 0.8 | 4.0 | 성공 가능성이 실패보다 4배 높은 상태임 |

---

# 3. 통계 및 머신러닝에서의 활용

## 📌 로지스틱 회귀 (Logistic Regression)
이진 분류 문제(0 또는 1)를 해결하기 위한 머신러닝 모델에서의 활용임.  
오즈는 0부터 무한대까지의 값을 가지는 형태로서 수식 계산에 적합한 구조임.

---

## 📌 로짓 (Logit / Log-Odds)
오즈에 자연로그(ln)를 적용한 값임.

log(Odds)

이 변환을 통해 확률 값을 -∞ ~ +∞ 범위로 확장하여 선형 모델로 표현하는 구조임.

---

## 📌 오즈비 (Odds Ratio)
두 집단 간 오즈를 비교하는 지표임.  
특정 요인이 결과에 미치는 영향 분석에 활용되는 지표임.

---

# 4. 핵심 정리

- 확률: 전체 대비 사건 발생 비율 개념임  
- 오즈: 성공과 실패의 비율 개념임  
- 오즈: 머신러닝 모델에서 선형화 가능한 구조임  
- 로지스틱 회귀: 오즈 기반 확률 예측 모델 구조임  